# DataMate v2 — Qwen2.5-Coder-14B-Instruct (4-bit QLoRA / BitsAndBytes)
**Kiến trúc:** Schema-Aware Code-Act Agent + Sandbox Execution (Timeout 10s) + Self-Correction (1 vòng) + Unanswerable/Abstain Gates  
**Phần cứng mục tiêu:** Google Colab GPU L4 (24GB VRAM) / 2x RTX 5090  
**Mục tiêu tối thượng:** Tối ưu hóa điểm số Penalized Accuracy $PA = \frac{\text{Đúng} - \text{Sai}}{N}$ (+1 Đúng, -1 Sai, 0 Abstain).

In [ ]:
# 1. Cài đặt các thư viện cần thiết trên Google Colab GPU L4
!pip install -q -U torch transformers accelerate bitsandbytes peft rapidfuzz openpyxl pandas tqdm

In [ ]:
# 2. Kiểm tra GPU & thiết lập Seed cố định (Deterministic 100% theo luật thi)
import os
import json
import time
import torch
from tqdm.auto import tqdm

os.environ["PYTHONHASHSEED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print(f"GPU khả dụng: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Cảnh báo: Đang chạy trên CPU!")

In [ ]:
# 3. Giải nén gói mã nguồn main_src và kiểm tra cấu trúc dữ liệu
import sys

# Nếu tải main_src.zip lên Colab, tự động giải nén:
if os.path.exists("main_src.zip") and not os.path.exists("main_src"):
    !unzip -q -o main_src.zip -d main_src

if os.path.abspath(".") not in sys.path:
    sys.path.insert(0, os.path.abspath("."))

from main_src.pipeline import DataMatePipeline
from main_src.formatters import dumps_submission_record, make_submission_record
from main_src.agent import AnswerSpecInferer, AnswerabilityGate
from main_src.sandbox import extract_code
print("Đã nạp thành công gói mã nguồn main_src!")

In [ ]:
# 4. Khởi tạo DataMatePipeline với Qwen/Qwen2.5-Coder-14B-Instruct (BitsAndBytes 4-bit NF4)
DATA_ROOT = "/content/public_test"  # Thay bằng đường dẫn tới thư mục chứa questions.jsonl, calendar.csv, data/
OUTPUT_SUBMISSION = "/content/submission.jsonl"
LOG_DIR = "/content/logs"
MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"
USE_LLM = True  # Đặt True trên GPU L4 để kích hoạt Qwen2.5-Coder-14B-Instruct 4-bit
BATCH_SIZE = 8  # Batch size tối ưu cho GPU L4 24GB VRAM

pipeline = DataMatePipeline(
    data_root_dir=DATA_ROOT,
    log_dir=LOG_DIR,
    use_llm=USE_LLM,
    llm_provider="transformers",
    llm_model=MODEL_ID,
    load_in_4bit=True,
    sandbox_timeout_s=10.0,
)

In [ ]:
# 5. Thực thi Batch Inference có thanh tiến trình tqdm, Self-Correction 1 vòng và tự động lưu submission.jsonl
def run_colab_batch_inference(pipe: DataMatePipeline, questions_path: str, output_path: str, batch_size: int = 8):
    with open(questions_path, "r", encoding="utf-8") as f:
        questions = [json.loads(line) for line in f if line.strip()]

    # Nhóm câu hỏi theo cửa hàng để tận dụng tối đa bộ nhớ đệm DataFrame (ShopBundle Cache)
    pending_llm_jobs = []
    final_records = {}

    print(f"Bước A: Quét {len(questions)} câu hỏi qua Preprocessor + TimeResolver + UnanswerableGate + FastPath Sandbox...")
    for q in tqdm(questions, desc="Pre-screening & Fast-Path"):
        qid = q["id"]
        biz_id = q["business_id"]
        bundle = pipe.get_shop_bundle(biz_id)
        if bundle is None:
            final_records[qid] = make_submission_record(qid, "abstain", None)
            continue

        pipe.preprocessor.set_shop_vocab(pipe.entity_matcher.extract_shop_vocab(bundle.entity_index))
        nq = pipe.preprocessor.normalize(q["question"])
        tctx = pipe.time_resolver.resolve(nq, q["as_of_date"])
        ents = pipe.entity_matcher.match(nq, bundle.entity_index)
        spec = AnswerSpecInferer.infer(nq, tctx)

        verdict, reason, _ = AnswerabilityGate.check(nq, tctx, bundle.profile)
        if verdict == "unanswerable":
            final_records[qid] = make_submission_record(qid, "unanswerable", None, reason=reason)
            continue
        if tctx.mode == "none":
            final_records[qid] = make_submission_record(qid, "abstain", None)
            continue

        det_code = pipe.agent.synthesize_deterministic_code(nq, bundle, tctx, ents, spec)
        if det_code:
            det_res = pipe.sandbox.execute(det_code, bundle, tctx, ents, pipe.cal_resolver, q["as_of_date"])
            if pipe.agent.is_valid_execution_result(det_res, spec, nq):
                final_records[qid] = make_submission_record(
                    qid, spec.ans_type, det_res.value, is_currency=spec.is_currency, month_only=spec.month_only
                )
                continue

        if pipe.agent.cfg.provider.lower() != "heuristic_only":
            pending_llm_jobs.append((q, nq, bundle, tctx, ents, spec))
        else:
            final_records[qid] = make_submission_record(qid, "abstain", None)

    # Bước B: Batch Inference trên Qwen2.5-Coder-14B-Instruct cho các câu phức tạp
    if pending_llm_jobs:
        print(f"Bước B: Chạy Batch Inference Qwen-14B cho {len(pending_llm_jobs)} câu hỏi phức tạp (batch_size={batch_size})...")
        for start_i in tqdm(range(0, len(pending_llm_jobs), batch_size), desc="Qwen-14B Batch Gen"):
            batch = pending_llm_jobs[start_i : start_i + batch_size]
            batch_msgs = []
            for q, nq, bundle, tctx, ents, spec in batch:
                u_prompt = pipe.agent.build_user_prompt(nq, bundle, tctx, ents, spec, q["as_of_date"])
                batch_msgs.append([
                    {"role": "system", "content": pipe.agent.SYSTEM_PROMPT},
                    {"role": "user", "content": u_prompt},
                ])

            outs_r1 = pipe.agent.generate_batch_llm_responses(batch_msgs)
            retry_jobs = []
            retry_msgs = []

            for job, msgs, out1 in zip(batch, batch_msgs, outs_r1):
                q, nq, bundle, tctx, ents, spec = job
                qid = q["id"]
                code1 = extract_code(out1 or "")
                res1 = pipe.sandbox.execute(code1, bundle, tctx, ents, pipe.cal_resolver, q["as_of_date"])
                if res1.unanswerable_reason is not None:
                    final_records[qid] = make_submission_record(qid, "unanswerable", None, reason=res1.unanswerable_reason)
                elif pipe.agent.is_valid_execution_result(res1, spec, nq):
                    final_records[qid] = make_submission_record(
                        qid, spec.ans_type, res1.value, is_currency=spec.is_currency, month_only=spec.month_only
                    )
                else:
                    tb_fb = res1.traceback if not res1.ok else f"Kết quả rỗng hoặc không hợp lệ ({repr(res1.value)})"
                    fb_str = f"Mã gặp lỗi [{tb_fb}]. Hãy quan sát lại Schema và sửa lại đoạn code!"
                    retry_jobs.append(job)
                    retry_msgs.append(msgs + [
                        {"role": "assistant", "content": out1 or ""},
                        {"role": "user", "content": fb_str},
                    ])

            # Vòng 2: Self-Correction duy nhất 1 lần cho các câu lỗi ở Vòng 1
            if retry_jobs:
                outs_r2 = pipe.agent.generate_batch_llm_responses(retry_msgs)
                for job, out2 in zip(retry_jobs, outs_r2):
                    q, nq, bundle, tctx, ents, spec = job
                    qid = q["id"]
                    code2 = extract_code(out2 or "")
                    res2 = pipe.sandbox.execute(code2, bundle, tctx, ents, pipe.cal_resolver, q["as_of_date"])
                    if res2.unanswerable_reason is not None:
                        final_records[qid] = make_submission_record(qid, "unanswerable", None, reason=res2.unanswerable_reason)
                    elif pipe.agent.is_valid_execution_result(res2, spec, nq):
                        final_records[qid] = make_submission_record(
                            qid, spec.ans_type, res2.value, is_currency=spec.is_currency, month_only=spec.month_only
                        )
                    else:
                        # Chốt chặn an toàn: trả về abstain để bảo toàn 0 điểm
                        final_records[qid] = make_submission_record(qid, "abstain", None)

    # Ghi file submission.jsonl theo đúng thứ tự câu hỏi gốc
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as out_f:
        for q in questions:
            rec = final_records.get(q["id"], make_submission_record(q["id"], "abstain", None))
            out_f.write(dumps_submission_record(rec) + "\n")

    print(f"Hoàn tất! Đã lưu {len(questions)} câu trả lời vào {output_path}")

if os.path.exists(os.path.join(DATA_ROOT, "questions.jsonl")):
    run_colab_batch_inference(pipeline, os.path.join(DATA_ROOT, "questions.jsonl"), OUTPUT_SUBMISSION, batch_size=BATCH_SIZE)